# BM25 on MS MARCO passages

This notebook reimplements the BM25 strategy from `configs/bm25.yml` on the MS MARCO passage dataset. We build the indices step by step, wrap scoring in a `SearchStrategy`, run `run_strategy`, and summarize NDCG/MRR.

Config values:
- k1 = 1.2
- b = 0.75
- title_boost = 9.3
- description_boost = 4.1

In [ ]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git@73ae7d4cd80f
from cheat_at_search.data_dir import mount
from cheat_at_search import data_dir

mount(use_gdrive=True)    # colab, share data across notebook runs on gdrive
# mount(use_gdrive=False) # <- colab without gdrive
# mount(use_gdrive=False, manual_path="/path/to/directory")  # <- force data path to specific directory, ie you're running locally.

data_dir.DATA_PATH

## Load MS MARCO corpus and judgments

The dataset loader will download the MS MARCO passages and small dev qrels on first use.

In [ ]:
import numpy as np
import pandas as pd

from cheat_at_search.msmarco_data import corpus, judgments

corpus = corpus.reset_index(drop=True)
doc_id_lookup = corpus["doc_id"].to_numpy()

corpus[["doc_id", "description"]].head(3)

## Tokenize a sample query

BM25 here is driven by the snowball stemmer tokenizer from cheat-at-search.

In [ ]:
from cheat_at_search.tokenizers import snowball_tokenizer

sample_query = judgments.iloc[0]["query"]
sample_query, snowball_tokenizer(sample_query)

## Build SearchArray indices

We index both title and description even though MS MARCO passages only contain description.

In [ ]:
from searcharray import SearchArray

title_index = SearchArray.index(
    corpus["title"].fillna(""),
    tokenizer=snowball_tokenizer,
    workers=4,
)
description_index = SearchArray.index(
    corpus["description"].fillna(""),
    tokenizer=snowball_tokenizer,
    workers=4,
)

description_index[0]

## Implement a BM25 SearchStrategy

We score each query against title and description using BM25, then combine the scores with the configured boosts.

In [ ]:
from searcharray.similarity import bm25_similarity
from cheat_at_search.strategy.strategy import SearchStrategy

class BM25Strategy(SearchStrategy):
    def __init__(
        self,
        corpus,
        title_index,
        description_index,
        k1=1.2,
        b=0.75,
        title_boost=9.3,
        description_boost=4.1,
        top_k=10,
        workers=4,
    ):
        super().__init__(corpus, top_k=top_k, workers=workers)
        self.title_index = title_index
        self.description_index = description_index
        self.title_boost = title_boost
        self.description_boost = description_boost
        self.similarity = bm25_similarity(k1=k1, b=b)

    def search(self, query, k=10):
        tokens = snowball_tokenizer(query)
        if not tokens:
            return np.array([], dtype=int), np.array([], dtype=float)

        title_scores = self.title_index.score(tokens, similarity=self.similarity)
        description_scores = self.description_index.score(tokens, similarity=self.similarity)
        scores = self.title_boost * title_scores + self.description_boost * description_scores

        top_k = min(k, len(scores))
        if top_k == 0:
            return np.array([], dtype=int), np.array([], dtype=float)

        top_idx = np.argpartition(-scores, top_k - 1)[:top_k]
        top_sorted = top_idx[np.argsort(-scores[top_idx])]
        return top_sorted, scores[top_sorted]

bm25 = BM25Strategy(corpus, title_index, description_index, top_k=10, workers=4)
bm25.search(sample_query, k=3)

## Run BM25 over all queries

Scoring against the full MS MARCO collection is expensive. If you want a faster iteration loop, set `NUM_QUERIES` to a smaller number.

We disable caching in the notebook to keep the workflow simple and transparent.

In [ ]:
from cheat_at_search.search import run_strategy

NUM_QUERIES = None  # set to e.g. 200 for faster iteration

graded = run_strategy(
    bm25,
    judgments,
    num_queries=NUM_QUERIES,
    seed=42,
    show_progress=True,
    cache=False,
)

graded.head()

## Summarize metrics

In [ ]:
from cheat_at_search.search import ndcgs, mrrs

ndcg_series = ndcgs(graded)
mrr_series = mrrs(graded)

pd.DataFrame(
    {
        "metric": ["NDCG@10", "MRR"],
        "value": [ndcg_series.mean(), mrr_series.mean()],
    }
)